# Solve a new problem with an existing algorithm

**Scenario:** you want to solve a problem that is not part of `qrisp_optimization.problems`
yet, using an algorithm that already exists, such as [`QAOA`][qrisp_optimization.QAOA].

The recipe is always the same:

1. find the requirements interface of the algorithm,
2. implement a problem class that satisfies it,
3. hand an instance to the algorithm.

As an example we implement **number partitioning**: split a multiset of numbers into two
groups whose sums are as close as possible.

In [1]:
import itertools
from collections.abc import Generator

import numpy as np
from numpy.typing import NDArray
from qrisp import QuantumArray, cx, h, p
from qrisp.qaoa import RX_mixer

from qrisp_optimization import (
    LRQAOA,
    QAOA,
    Bruteforce,
    BruteforceRequirements,
    QAOARequirements,
)
from qrisp_optimization.utils import QuantumBinary

## 1. Check the algorithm's requirements

Every algorithm states in its signature which interface it consumes:
`QAOA(problem: QAOARequirements, ...)`, `Bruteforce(problem: BruteforceRequirements)`.
The abstract methods of that interface are exactly what you have to implement:

In [2]:
print("QAOARequirements:      ", sorted(QAOARequirements.__abstractmethods__))
print("BruteforceRequirements:", sorted(BruteforceRequirements.__abstractmethods__))

QAOARequirements:       ['cl_cost_function', 'cost_layer', 'mixer_layer', 'state_prep']
BruteforceRequirements: ['cl_cost_function', 'next']


Both inherit `cl_cost_function` from
[`CLCostRequirements`][qrisp_optimization.CLCostRequirements], so a problem implementing both
interfaces needs five methods in total. If you leave one out, Python refuses to instantiate
the class:

In [3]:
class Incomplete(QAOARequirements):
    def cl_cost_function(self, x) -> float:
        return 0.0


try:
    Incomplete()
except TypeError as e:
    print(e)

Can't instantiate abstract class Incomplete without an implementation for abstract methods 'cost_layer', 'mixer_layer', 'state_prep'


## 2. Formulate the problem

Given numbers $a_1, \dots, a_n$, a binary vector $x$ assigns $a_i$ to group $x_i$. With
spins $s_i = 1 - 2x_i \in \{\pm 1\}$, the difference of the two group sums is
$\sum_i a_i s_i$, so we minimize

$$
C(x) = \Big(\sum_i a_i s_i\Big)^2
     = \sum_i a_i^2 + 2 \sum_{i<j} a_i a_j\, s_i s_j .
$$

$C(x) = 0$ exactly for a perfect partition.

For the QAOA cost layer we need $e^{-i\gamma C}$. Using $s_i s_j = 1 - 2(x_i \oplus x_j)$ and
dropping global phases, this becomes

$$
e^{-i\gamma C} \simeq \prod_{i<j} e^{\,4 i \gamma a_i a_j (x_i \oplus x_j)},
$$

i.e. one phase gate per pair, applied to the parity $x_i \oplus x_j$. That parity is computed
into qubit $j$ with a `cx`, phased with `p`, and uncomputed with a second `cx` (the same
pattern [`MaxCutProblem`][qrisp_optimization.problems.MaxCutProblem] uses).

Like [`QUBOProblem`][qrisp_optimization.problems.QUBOProblem], we rescale the angle, here by
$\sum_i a_i^2$, so that typical starting angles around `0.5` work independently of the size
of the numbers.

## 3. Implement the problem class

State preparation and mixer are the standard choices for unconstrained binary problems:
a Hadamard on every qubit and an `RX` mixer.

In [4]:
type BinaryIntArray = NDArray[np.int_]


class NumberPartitioningProblem(
    QAOARequirements[BinaryIntArray], BruteforceRequirements[BinaryIntArray]
):
    # Number partitioning: minimize (sum_i a_i s_i)^2 with s_i = 1 - 2 x_i.

    def __init__(self, a: list[float]) -> None:
        self.a = np.asarray(a, dtype=float)
        self.n = len(self.a)
        self.norm = float(np.sum(self.a**2))

    # needed for QAOA and Bruteforce
    def cl_cost_function(self, x: BinaryIntArray) -> float:
        s = 1 - 2 * np.asarray(x)
        return float(np.dot(self.a, s) ** 2)

    # needed for Bruteforce only
    def next(self) -> Generator[BinaryIntArray]:
        for bits in itertools.product((0, 1), repeat=self.n):
            yield np.array(bits, dtype=int)

    # needed for QAOA only
    def state_prep(self, qarg: QuantumArray) -> QuantumArray:
        for qv in qarg:
            h(qv)
        return qarg

    # needed for QAOA only
    def cost_layer(self, qarg: QuantumArray, gamma: float) -> QuantumArray:
        for i, j in itertools.combinations(range(self.n), 2):
            cx(qarg[i], qarg[j])
            p(4 * gamma * self.a[i] * self.a[j] / self.norm, qarg[j])
            cx(qarg[i], qarg[j])
        return qarg

    # needed for QAOA only
    def mixer_layer(self, qarg: QuantumArray, beta: float) -> QuantumArray:
        for qv in qarg:
            RX_mixer(qv, beta)
        return qarg

!!! note "Only one interface?"
    You don't have to implement every interface. If enumerating your search space makes no
    sense, implement only `QAOARequirements`: `QAOA` and `LRQAOA` will accept the problem,
    and a type checker will flag any attempt to pass it to `Bruteforce`.

## 4. Instantiate the problem

In [5]:
problem = NumberPartitioningProblem([4, 7, 2, 5, 3, 1])

print("sum:", problem.a.sum())
print("cost of x = 000111:", problem.cl_cost_function(np.array([0, 0, 0, 1, 1, 1])))

sum: 22.0
cost of x = 000111: 16.0


## 5. Solve it with the existing algorithms

First the classical reference. Note that a solution and its complement (all bits
flipped) describe the same partition, so optima always come in pairs.

In [6]:
optimal = Bruteforce(problem).solve()
optimal_set = {tuple(x) for x in optimal}

print(f"{len(optimal)} optimal solutions with cost {problem.cl_cost_function(optimal[0])}")
for x in optimal:
    print(x, "->", problem.a[x == 0], problem.a[x == 1])

4 optimal solutions with cost 0.0
[0 0 1 1 1 1] -> [4. 7.] [2. 5. 3. 1.]
[0 1 0 0 1 1] -> [4. 2. 5.] [7. 3. 1.]
[1 0 1 1 0 0] -> [7. 3. 1.] [4. 2. 5.]
[1 1 0 0 0 0] -> [2. 5. 3. 1.] [4. 7.]


Now QAOA, exactly as in [Solve an existing problem](solve-existing-problem.ipynb).
Nothing in `QAOA` knows about number partitioning.

Number partitioning has a rugged cost landscape, so low-depth QAOA only modestly beats
random guessing here, and small initial angles work noticeably better than `0.5`. How well
an algorithm does is a property of the problem *and* the algorithm. The point is that
trying one costs a single line.

In [7]:
def report(meas: dict, top: int = 5) -> None:
    p_opt = sum(prob for x, prob in meas.items() if tuple(x) in optimal_set)
    p_random = len(optimal_set) / 2**problem.n
    print(f"P(optimal) = {p_opt:.3f}   (random guessing: {p_random:.3f})")
    for x, prob in sorted(meas.items(), key=lambda kv: -kv[1])[:top]:
        marker = "  <- optimal" if tuple(x) in optimal_set else ""
        print(f"  {x}  cost={problem.cl_cost_function(np.array(x)):6.1f}  p={prob:.3f}{marker}")


qaoa = QAOA(problem=problem, qarg=lambda: QuantumArray(qtype=QuantumBinary(), shape=problem.n))
theta = qaoa.optimize(np.full(6, 0.1), max_iter=200)
report(qaoa.ansatz(theta).get_measurement())

P(optimal) = 0.139   (random guessing: 0.062)
  [0 1 0 1 0 0]  cost=   4.0  p=0.046
  [1 0 1 0 1 1]  cost=   4.0  p=0.046
  [0 1 0 1 0 1]  cost=  16.0  p=0.038
  [1 0 1 0 1 0]  cost=  16.0  p=0.038
  [0 0 1 1 1 0]  cost=   4.0  p=0.036


In [8]:
lr_qaoa = LRQAOA(problem=problem, qarg=lambda: QuantumArray(qtype=QuantumBinary(), shape=problem.n))
report(lr_qaoa.ansatz(depth=10, d_gamma=1.0, d_beta=0.6).get_measurement())

P(optimal) = 0.165   (random guessing: 0.062)
  [0 0 1 1 1 1]  cost=   0.0  p=0.041  <- optimal
  [1 1 0 0 0 0]  cost=   0.0  p=0.041  <- optimal
  [0 1 1 0 0 1]  cost=   4.0  p=0.041
  [1 0 0 1 1 0]  cost=   4.0  p=0.041
  [0 1 0 0 1 1]  cost=   0.0  p=0.041  <- optimal


## Adding the problem to the package

Once the problem works in a notebook, move it into the library:

1. Create `qrisp_optimization/problems/number_partitioning_problem.py` with the class,
   with NumPy-style docstrings like the existing problems.
2. Export it in `qrisp_optimization/problems/__init__.py`.
3. Add `tests/test_number_partitioning_problem.py`. The existing problem tests are a good
   template; at least check `cl_cost_function` on hand-computed cases, that `next` yields
   all $2^n$ vectors exactly once, and that QAOA's most likely outcome matches `Bruteforce`
   on a small instance.
4. Run `scripts/check.sh` (ruff, mypy, pytest).

The API reference page is generated automatically from the module.